In [1]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path

Mounted at /content/drive


In [2]:
ssh_path=Path('/content/drive/MyDrive/Colab Notebooks/DLBasics2023_colab/Competition/.ssh/')
!mkdir -p "$ssh_path"
ssh_key_path=ssh_path / "id_rsa"
print(str(ssh_key_path))
repo_path=Path("/content/dl_lecture_competition_private")
print(str(repo_path))
content_path=Path("/content")

/content/drive/MyDrive/Colab Notebooks/DLBasics2023_colab/Competition/.ssh/id_rsa
/content/dl_lecture_competition_private


# clone repository

In [3]:
!mkdir -p /root/.ssh
!chmod 600 /root/.ssh
!ssh-keyscan -t rsa github.com >> /root/.ssh/known_hosts
!cp "$ssh_key_path" /root/.ssh/id_rsa
!git config --global user.name "Nishimura Naoto"
!git config --global user.email "21977496+mptnan@users.noreply.github.com"
%cd "$content_path"
!rm -rf "$repo_path"
!git clone git@github.com:mptnan/dl_lecture_competition_private.git "$repo_path"
%cd "$repo_path"
!git checkout dev_vqa

# github.com:22 SSH-2.0-babeld-2533c54da
/content
Cloning into '/content/dl_lecture_competition_private'...
remote: Enumerating objects: 612, done.
remote: Counting objects: 100% (153/153), done.
remote: Compressing objects: 100% (104/104), done.
remote: Total 612 (delta 104), reused 97 (delta 49), pack-reused 459
Receiving objects: 100% (612/612), 259.09 KiB | 1012.00 KiB/s, done.
Resolving deltas: 100% (361/361), done.
/content/dl_lecture_competition_private
Branch 'dev_vqa' set up to track remote branch 'dev_vqa' from 'origin'.
Switched to a new branch 'dev_vqa'


# prepare data 〜 5 mins w/ TPU

In [4]:
data_path=repo_path / "data"
data_path_src=Path('/content/drive/MyDrive/Colab Notebooks/DLBasics2023_colab/Competition/vqa/VQA')
data_train_zip_src=data_path_src / "train.zip"
data_valid_zip_src=data_path_src / "valid.zip"
data_train_json_src=data_path_src / "train.json"
data_valid_json_src=data_path_src / "valid.json"

In [5]:
%pushd "$data_path"
!unzip "$data_train_zip_src" # ~3 mins
!unzip "$data_valid_zip_src" # ~1 min
!cp "$data_train_json_src" .
!cp "$data_valid_json_src" .
%popd

ストリーミング出力は最後の 5000 行に切り捨てられました。
  inflating: train/train_09178.jpg   
  inflating: train/train_04053.jpg   
  inflating: train/train_18288.jpg   
  inflating: train/train_04604.jpg   
  inflating: train/train_07889.jpg   
  inflating: train/train_07074.jpg   
  inflating: train/train_01697.jpg   
  inflating: train/train_17744.jpg   
  inflating: train/train_11013.jpg   
  inflating: train/train_18948.jpg   
  inflating: train/train_14576.jpg   
  inflating: train/train_14138.jpg   
  inflating: train/train_19228.jpg   
  inflating: train/train_00291.jpg   
  inflating: train/train_13361.jpg   
  inflating: train/train_08819.jpg   
  inflating: train/train_10060.jpg   
  inflating: train/train_01538.jpg   
  inflating: train/train_09467.jpg   
  inflating: train/train_10874.jpg   
  inflating: train/train_11688.jpg   
  inflating: train/train_05628.jpg   
  inflating: train/train_03379.jpg   
  inflating: train/train_06414.jpg   
  inflating: train/train_14175.jpg   
  inflating: train

# main 〜 10 mins / epoch w/ TPU

In [6]:
!pip install -r requirements_gcolab.txt # cancel warning

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.5/154.5 kB 4.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 6.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for antlr4-python3-runtime: filename=antlr4_python3_runtime-4.9.3-py3-none-any.whl size=144554 sha256=dd332b14a586f612f71d9b4f6bf055875caf178f0e5143a1904f1f3bd42edb66
  Stored in directory: /root/.cache/pip/wheels/12/93/dd/1f6a127edc45659556564c5730f6d4e300888f4bca2d4c5a88
Successfully built antlr4-python3-runtime


In [50]:
!git checkout dev_vqa

Switched to branch 'dev_vqa'
Your branch is up to date with 'origin/dev_vqa'.


In [55]:
!git reset --hard
!git pull

HEAD is now at 642b36d again bert embedding model
remote: Enumerating objects: 3, done.
remote: Counting objects: 100% (3/3), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0
Unpacking objects: 100% (3/3), 322 bytes | 322.00 KiB/s, done.
From github.com:mptnan/dl_lecture_competition_private
   642b36d..35e8643  dev_vqa    -> origin/dev_vqa
Updating 642b36d..35e8643
Fast-forward
 main.py | 6 ++----
 1 file changed, 2 insertions(+), 4 deletions(-)


In [57]:
# tpuは使わない
!python main_bert_question_onehot_answer.py env=gcolab env.num_epoch=15 env.num_workers=2 env.lr=0.002

/usr/local/lib/python3.10/dist-packages/torchtext/data/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/vocab/__init__.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated and the last released version will be 0.18 (this one). You can silence this warning by calling the following at the beginnign of your scripts: `import torchtext; torchtext.disable_torchtext_deprecation_warning()`
  warnings.warn(torchtext._TORCHTEXT_DEPRECATION_MSG)
/usr/local/lib/python3.10/dist-packages/torchtext/utils.py:4: UserWarning: 
/!\ IMPORTANT WARNING ABOUT TORCHTEXT STATUS /!\ 
Torchtext is deprecated